First, let's load the `application_train.csv` file into a pandas DataFrame. If you haven't uploaded the files yet, please do so now.

In [ ]:
import pandas as pd

# Load the main training dataset
try:
    df_train = pd.read_csv('application_train.csv')
    print("application_train.csv loaded successfully.")
except FileNotFoundError:
    print("Error: 'application_train.csv' not found. Please upload the file or ensure the path is correct.")
    # Exit or handle the error appropriately, e.g., prompt user to upload
    df_train = None # Set to None to avoid further errors


```mermaid
graph TD
    A[Start: Data Input] --> B{Feature Selection};
    B --> B1[Load application_train.csv];
    B1 --> B2[Initial Features: All columns];
    B2 --> B3[Drop SK_ID_CURR];

    B3 --> C{Data Preprocessing};
    C --> C1[Missing Values Handling];
    C1 --> C1_1[Columns with >50% missing values removed];
    C1 --> C1_2[Numerical missing values imputed with Median];
    C1 --> C1_3[Categorical missing values imputed with Mode];

    C1_3 --> C2[Categorical Variables];
    C2 --> C2_1[One-Hot Encoding];

    C2_1 --> C3[Data Splitting];
    C3 --> C3_1[Separate Features X and Target y];
    C3 --> C3_2[Split into Train 80% and Test 20%];
    C3 --> C3_3[Stratified Split for Target Class Balance];

    C3_3 --> D{Modeling Algorithms};
    D --> D1[Logistic Regression];
    D1 --> D1_1[Imbalance Handling: class_weight='balanced'];

    D --> D2[XGBoost Classifier];
    D2 --> D2_1[Imbalance Handling: scale_pos_weight];

    D --> D3[Decision Tree Classifier];
    D3 --> D3_1[Imbalance Handling: class_weight='balanced'];

    D1_1 & D2_1 & D3_1 --> E{Evaluation Metrics};
    E --> E1[ROC-AUC Score];
    E1 --> F[End];
```

```mermaid
graph TD
    A[Start: Data Input] --> B{Feature Selection};
    B --> B1[Load application_train.csv];
    B1 --> B2[Initial Features: All columns];
    B2 --> B3[Drop SK_ID_CURR];

    B3 --> C{Data Preprocessing};
    C --> C1[Missing Values Handling];
    C1 --> C1_1[Columns with >50% missing values removed];
    C1 --> C1_2[Numerical missing values imputed with Median];
    C1 --> C1_3[Categorical missing values imputed with Mode];

    C1_3 --> C2[Categorical Variables];
    C2 --> C2_1[One-Hot Encoding];

    C2_1 --> C3[Data Splitting];
    C3 --> C3_1[Separate Features X and Target y];
    C3 --> C3_2[Split into Train 80% and Test 20%];
    C3 --> C3_3[Stratified Split for Target Class Balance];

    C3_3 --> D{Modeling Algorithms};
    D --> D1[Logistic Regression];
    D1 --> D1_1[Imbalance Handling: class_weight='balanced'];

    D --> D2[XGBoost Classifier];
    D2 --> D2_1[Imbalance Handling: scale_pos_weight];

    D --> D3[Decision Tree Classifier];
    D3 --> D3_1[Imbalance Handling: class_weight='balanced'];

    D1_1 & D2_1 & D3_1 --> E{Evaluation Metrics};
    E --> E1[ROC-AUC Score];
    E1 --> F[End];
```

**Cara menyimpan sebagai file:**
1.  Salin seluruh teks di dalam blok kode di atas (mulai dari `graph TD` hingga `F[End];`).
2.  Buka editor teks apa pun di komputer Anda (seperti Notepad di Windows, TextEdit di macOS, atau editor kode seperti VS Code).
3.  Tempel teks yang sudah disalin.
4.  Simpan file dengan nama seperti `alur_pemodelan.mmd` atau `alur_pemodelan.txt`.

## 1. Data Loading

In [ ]:
import pandas as pd

# Load the main training dataset
try:
    df_train = pd.read_csv('application_train.csv')
    print("application_train.csv loaded successfully.")
except FileNotFoundError:
    print("Error: 'application_train.csv' not found. Please upload the file or ensure the path is correct.")
    df_train = None # Set to None to avoid further errors

## 2. Initial Data Exploration

In [ ]:
if df_train is not None:
    print("\n--- First 5 rows of application_train.csv ---")
    display(df_train.head())

    print("\n--- Dataset Information (Data Types and Non-Null Counts) ---")
    df_train.info()

    print("\n--- Missing Values Count ---")
    missing_values = df_train.isnull().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
    display(pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage': (missing_values / len(df_train)) * 100}))

    print("\n--- Descriptive Statistics ---")
    display(df_train.describe())

    print("\n--- Target Variable Distribution ---")
    display(df_train['TARGET'].value_counts(normalize=True) * 100)

## 3. Data Preprocessing

### 3.1. Handling Missing Values

In [ ]:
# Menghitung persentase nilai hilang untuk setiap kolom
missing_percentage = df_train.isnull().sum() / len(df_train) * 100

# Menampilkan kolom dengan persentase nilai hilang lebih dari ambang batas (misal 50%)
threshold = 50
columns_to_drop = missing_percentage[missing_percentage > threshold].index.tolist()

print(f"Kolom yang akan dihapus (nilai hilang > {threshold}%): {len(columns_to_drop)} kolom")
print(columns_to_drop)

# Menghapus kolom-kolom tersebut dari DataFrame
df_train_interim = df_train.drop(columns=columns_to_drop)
print(f"Jumlah kolom setelah penghapusan: {df_train_interim.shape[1]}")

# Memperbarui daftar kolom yang memiliki nilai hilang
missing_values_after_drop = df_train_interim.isnull().sum()
missing_values_after_drop = missing_values_after_drop[missing_values_after_drop > 0].sort_values(ascending=False)

print("\n--- Nilai Hilang Setelah Penghapusan Kolom --- ")
display(pd.DataFrame({'Missing Count': missing_values_after_drop, 'Missing Percentage': (missing_values_after_drop / len(df_train_interim)) * 100}))

# Memisahkan kolom numerik dan kategorikal yang masih memiliki nilai hilang
numerical_cols_with_missing = df_train_interim.select_dtypes(include=['int64', 'float64']).columns
categorical_cols_with_missing = df_train_interim.select_dtypes(include=['object']).columns

# Imputasi kolom numerik dengan median
for col in numerical_cols_with_missing:
    if df_train_interim[col].isnull().any():
        median_val = df_train_interim[col].median()
        df_train_interim[col] = df_train_interim[col].fillna(median_val)

# Imputasi kolom kategorikal dengan modus (mode)
for col in categorical_cols_with_missing:
    if df_train_interim[col].isnull().any():
        mode_val = df_train_interim[col].mode()[0]
        df_train_interim[col] = df_train_interim[col].fillna(mode_val)

print("\n--- Verifikasi Nilai Hilang Setelah Imputasi --- ")
missing_values_final = df_train_interim.isnull().sum()
missing_values_final = missing_values_final[missing_values_final > 0]

if missing_values_final.empty:
    print("Tidak ada lagi nilai hilang di DataFrame.")
else:
    display(pd.DataFrame({'Missing Count': missing_values_final}))

### 3.2. Handling Categorical Variables (One-Hot Encoding)

In [ ]:
# Identifikasi kolom kategorikal
categorical_cols = df_train_interim.select_dtypes(include=['object']).columns

print(f"Kolom kategorikal yang akan di-encode: {len(categorical_cols)} kolom")
print(categorical_cols.tolist())

# Lakukan One-Hot Encoding
df_train_processed = pd.get_dummies(df_train_interim, columns=categorical_cols, dummy_na=False)

print(f"\nJumlah kolom setelah One-Hot Encoding: {df_train_processed.shape[1]}")

# Tampilkan 5 baris pertama DataFrame setelah encoding
print("\n--- 5 Baris Pertama DataFrame Setelah One-Hot Encoding ---")
display(df_train_processed.head())

### 3.3. Preparing Data for Modeling (Splitting Features and Target)

In [ ]:
from sklearn.model_selection import train_test_split

# Memisahkan fitur (X) dan target (y)
X = df_train_processed.drop('TARGET', axis=1)
y = df_train_processed['TARGET']

# Menangani kolom non-numerik yang mungkin tersisa (misal, boolean)
# Konversi boolean menjadi integer (0 atau 1)
for col in X.select_dtypes(include=['bool']).columns:
    X[col] = X[col].astype(int)

# Hapus kolom 'SK_ID_CURR' karena tidak digunakan dalam pemodelan sebagai fitur
X = X.drop('SK_ID_CURR', axis=1, errors='ignore')

print(f"Jumlah fitur (X) setelah persiapan: {X.shape[1]}")
print(f"Ukuran variabel target (y): {y.shape[0]}")

# Membagi data menjadi set pelatihan dan pengujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Ukuran data pelatihan (X_train): {X_train.shape}")
print(f"Ukuran data pengujian (X_test): {X_test.shape}")
print(f"Distribusi target di data pelatihan:\n{y_train.value_counts(normalize=True)}")
print(f"Distribusi target di data pengujian:\n{y_test.value_counts(normalize=True)}")

## 4. Modeling

### 4.1. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Inisialisasi model Logistic Regression
# Menggunakan `class_weight='balanced'` untuk menangani ketidakseimbangan kelas
log_reg_model = LogisticRegression(solver='liblinear', random_state=42, class_weight='balanced', max_iter=1000)

# Melatih model pada data pelatihan
print("Melatih model Logistic Regression...")
log_reg_model.fit(X_train, y_train)
print("Model Logistic Regression selesai dilatih.")

# Melakukan prediksi probabilitas pada data pengujian
y_pred_log_reg = log_reg_model.predict_proba(X_test)[:, 1]

# Menghitung dan menampilkan ROC-AUC score
roc_auc_log_reg = roc_auc_score(y_test, y_pred_log_reg)
print(f"ROC-AUC Score (Logistic Regression): {roc_auc_log_reg:.4f}")

### 4.2. XGBoost Classifier

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Inisialisasi model XGBoost Classifier
# Menggunakan `scale_pos_weight` untuk menangani ketidakseimbangan kelas
# Nilai `scale_pos_weight` dihitung sebagai (jumlah_negatif / jumlah_positif)
positive_cases = y_train.sum()
negative_cases = len(y_train) - positive_cases
scale_pos_weight = negative_cases / positive_cases

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=200, # Jumlah estimator awal
    learning_rate=0.1,
    max_depth=5,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    scale_pos_weight=scale_pos_weight # Mengatasi ketidakseimbangan kelas
)

# Melatih model pada data pelatihan
print("Melatih model XGBoost...")
xgb_model.fit(X_train, y_train)
print("Model XGBoost selesai dilatih.")

# Melakukan prediksi probabilitas pada data pengujian
y_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Menghitung dan menampilkan ROC-AUC score
roc_auc_xgb = roc_auc_score(y_test, y_pred_xgb)
print(f"ROC-AUC Score (XGBoost): {roc_auc_xgb:.4f}")

### 4.3. Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Inisialisasi model Decision Tree Classifier
# Menggunakan `class_weight='balanced'` untuk menangani ketidakseimbangan kelas
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# Melatih model pada data pelatihan
print("Melatih model Decision Tree...")
dt_model.fit(X_train, y_train)
print("Model Decision Tree selesai dilatih.")

# Melakukan prediksi probabilitas pada data pengujian
y_pred_dt = dt_model.predict_proba(X_test)[:, 1]

# Menghitung dan menampilkan ROC-AUC score
roc_auc_dt = roc_auc_score(y_test, y_pred_dt)
print(f"ROC-AUC Score (Decision Tree): {roc_auc_dt:.4f}")

## 5. Model Evaluation Summary

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

print("Memulai evaluasi performa untuk setiap model...")

# --- Logistic Regression ---
# Predictions for Logistic Regression
y_train_pred_log_reg = log_reg_model.predict(X_train)
y_test_pred_log_reg = log_reg_model.predict(X_test)
y_test_pred_proba_log_reg = log_reg_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for Logistic Regression
train_accuracy_log_reg = accuracy_score(y_train, y_train_pred_log_reg) * 100
test_accuracy_log_reg = accuracy_score(y_test, y_test_pred_log_reg) * 100
roc_auc_log_reg_score = roc_auc_score(y_test, y_test_pred_proba_log_reg) * 100 # Convert to percentage

# --- XGBoost Classifier ---
# Predictions for XGBoost
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)
y_test_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for XGBoost
train_accuracy_xgb = accuracy_score(y_train, y_train_pred_xgb) * 100
test_accuracy_xgb = accuracy_score(y_test, y_test_pred_xgb) * 100
roc_auc_xgb_score = roc_auc_score(y_test, y_test_pred_proba_xgb) * 100 # Convert to percentage

# --- Decision Tree Classifier ---
# Predictions for Decision Tree
y_train_pred_dt = dt_model.predict(X_train)
y_test_pred_dt = dt_model.predict(X_test)
y_test_pred_proba_dt = dt_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for Decision Tree
train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt) * 100
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt) * 100
roc_auc_dt_score = roc_auc_score(y_test, y_test_pred_proba_dt) * 100 # Convert to percentage

print("Evaluasi performa selesai.")

# Create a DataFrame to display the results neatly
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost', 'Decision Tree'],
    'Training Accuracy (%)': [train_accuracy_log_reg, train_accuracy_xgb, train_accuracy_dt],
    'Testing Accuracy (%)': [test_accuracy_log_reg, test_accuracy_xgb, test_accuracy_dt],
    'ROC-AUC Score (%)': [roc_auc_log_reg_score, roc_auc_xgb_score, roc_auc_dt_score]
})

display(results_df)

## 6. Feature Importance from XGBoost Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Dapatkan feature importance dari model XGBoost
feature_importances = xgb_model.feature_importances_

# Buat DataFrame untuk visualisasi
features_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
})

# Urutkan berdasarkan importance
features_df = features_df.sort_values(by='Importance', ascending=False)

print("Top 20 Fitur Paling Penting menurut XGBoost:")
display(features_df.head(20))

# Visualisasi Top 20 Feature Importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df.head(20), hue='Feature', palette='viridis', legend=False)
plt.title('Top 20 Feature Importance dari Model XGBoost')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 7. Visualizing Key Insights

### 7.1. Insight 1: External Credit Quality and Employment Status

First, let's look at the distribution of `EXT_SOURCE_2` and `EXT_SOURCE_3` (external source scores) compared to the target variable (`TARGET`). Lower scores on `EXT_SOURCE` generally indicate higher risk.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 6))

# Plot EXT_SOURCE_2 vs TARGET
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.violinplot(x='TARGET', y='EXT_SOURCE_2', data=df_train_processed, hue='TARGET', palette='viridis', legend=False)
plt.title('EXT_SOURCE_2 Distribution by Target')
plt.xlabel('Target (0: Repaid, 1: Default)')
plt.ylabel('EXT_SOURCE_2 Score')

# Plot EXT_SOURCE_3 vs TARGET
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.violinplot(x='TARGET', y='EXT_SOURCE_3', data=df_train_processed, hue='TARGET', palette='plasma', legend=False)
plt.title('EXT_SOURCE_3 Distribution by Target')
plt.xlabel('Target (0: Repaid, 1: Default)')
plt.ylabel('EXT_SOURCE_3 Score')

plt.tight_layout()
plt.show()

Next, let's visualize the relationship between `NAME_INCOME_TYPE` (income type) and `FLAG_EMP_PHONE` (ownership of a mobile phone registered as an office phone) with the probability of default (`TARGET`). `FLAG_EMP_PHONE` indicates employment stability.

In [ ]:
plt.figure(figsize=(16, 6))

# Plot NAME_INCOME_TYPE vs TARGET (Mean of Target)
plt.subplot(1, 2, 1)
sns.barplot(x='NAME_INCOME_TYPE', y='TARGET', data=df_train_processed, hue='NAME_INCOME_TYPE', palette='coolwarm', legend=False)
plt.title('Default Rate by Income Type')
plt.xlabel('Income Type')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks(rotation=45, ha='right')

# Plot FLAG_EMP_PHONE vs TARGET (Mean of Target)
plt.subplot(1, 2, 2)
sns.barplot(x='FLAG_EMP_PHONE', y='TARGET', data=df_train_processed, hue='FLAG_EMP_PHONE', palette='rocket', legend=False)
plt.title('Default Rate by Employee Phone Flag (1: Has Phone, 0: No Phone)')
plt.xlabel('FLAG_EMP_PHONE')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks([0, 1], ['No Employee Phone', 'Has Employee Phone'])

plt.tight_layout()
plt.show()

### 7.2. Insight 2: Education and Consumer Demographics

Finally, let's visualize how education level, car ownership, and client region rating (`REGION_RATING_CLIENT_W_CITY`) are related to the probability of default (`TARGET`).

In [ ]:
plt.figure(figsize=(18, 6))

# Plot NAME_EDUCATION_TYPE vs TARGET (Mean of Target)
plt.subplot(1, 3, 1)
sns.barplot(x='NAME_EDUCATION_TYPE', y='TARGET', data=df_train_processed, hue='NAME_EDUCATION_TYPE', palette='crest', legend=False)
plt.title('Default Rate by Education Type')
plt.xlabel('Education Type')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks(rotation=45, ha='right')

# Plot FLAG_OWN_CAR vs TARGET (Mean of Target)
plt.subplot(1, 3, 2)
sns.barplot(x='FLAG_OWN_CAR', y='TARGET', data=df_train_processed, hue='FLAG_OWN_CAR', palette='mako', legend=False)
plt.title('Default Rate by Car Ownership')
plt.xlabel('Owns Car (N: No, Y: Yes)')
plt.ylabel('Default Rate (Mean Target)')

# Plot REGION_RATING_CLIENT_W_CITY vs TARGET (Mean of Target)
plt.subplot(1, 3, 3)
sns.barplot(x='REGION_RATING_CLIENT_W_CITY', y='TARGET', data=df_train_processed, hue='REGION_RATING_CLIENT_W_CITY', palette='rocket', legend=False)
plt.title('Default Rate by Region Rating (Client in City)')
plt.xlabel('Region Rating (Client in City)')
plt.ylabel('Default Rate (Mean Target)')

plt.tight_layout()
plt.show()

## 8. Workflow Summary (Mermaid and Table)

```mermaid
graph TD
    A[Start: Data Input] --> B{Feature Selection};
    B --> B1[Load application_train.csv];
    B1 --> B2[Initial Features: All columns];
    B2 --> B3[Drop SK_ID_CURR];

    B3 --> C{Data Preprocessing};
    C --> C1[Missing Values Handling];
    C1 --> C1_1[Columns with >50% missing values removed];
    C1 --> C1_2[Numerical missing values imputed with Median];
    C1 --> C1_3[Categorical missing values imputed with Mode];

    C1_3 --> C2[Categorical Variables];
    C2 --> C2_1[One-Hot Encoding];

    C2_1 --> C3[Data Splitting];
    C3 --> C3_1[Separate Features X and Target y];
    C3 --> C3_2[Split into Train 80% and Test 20%];
    C3 --> C3_3[Stratified Split for Target Class Balance];

    C3_3 --> D{Modeling Algorithms};
    D --> D1[Logistic Regression];
    D1 --> D1_1[Imbalance Handling: class_weight='balanced'];

    D --> D2[XGBoost Classifier];
    D2 --> D2_1[Imbalance Handling: scale_pos_weight];

    D --> D3[Decision Tree Classifier];
    D3 --> D3_1[Imbalance Handling: class_weight='balanced'];

    D1_1 & D2_1 & D3_1 --> E{Evaluation Metrics};
    E --> E1[ROC-AUC Score];
    E1 --> F[End];
```

**Cara menyimpan sebagai file:**
1.  Salin seluruh teks di dalam blok kode di atas (mulai dari `graph TD` hingga `F[End];`).
2.  Buka editor teks apa pun di komputer Anda (seperti Notepad di Windows, TextEdit di macOS, atau editor kode seperti VS Code).
3.  Tempel teks yang sudah disalin.
4.  Simpan file dengan nama seperti `alur_pemodelan.mmd` atau `alur_pemodelan.txt`.

### Ringkasan Alur Pembuatan Model (Tabel)

| Tahap                      | Sub-Tahap                         | Deskripsi Detail                                                                  |
| :------------------------- | :-------------------------------- | :-------------------------------------------------------------------------------- |
| **1. Data Input & Feature Selection** | **Sumber Data**                   | `application_train.csv`                                                           |
|                            | **Fitur Awal**                    | Semua kolom dari `application_train.csv`                                          |
|                            | **Penghapusan Fitur**             | Kolom `SK_ID_CURR` dihapus.                                                       |
| **2. Pra-pemrosesan Data** | **Penanganan Missing Values**     | Kolom dengan `>50%` missing values dihapus. Numerik diimputasi dengan Median. Kategorikal diimputasi dengan Modus. |
|                            | **Variabel Kategorikal**          | Dilakukan **One-Hot Encoding** pada 13 kolom kategorikal.                          |
|                            | **Pembagian Data**                | Fitur (`X`) dan Target (`y`) dipisah. Data dibagi 80% Train, 20% Test (Stratified Split). |
| **3. Algoritma Pemodelan** | **Logistic Regression**           | Menggunakan `class_weight='balanced'`. ROC-AUC: 0.6425.                          |
|                            | **XGBoost Classifier**            | Menggunakan `scale_pos_weight`. ROC-AUC: 0.7567 (terbaik).                        |
|                            | **Decision Tree Classifier**      | Menggunakan `class_weight='balanced'`. ROC-AUC: 0.5429.                          |
| **4. Metrik Evaluasi**     | **ROC-AUC Score**                 | Digunakan untuk semua model sebagai metrik utama.                                 |

**Cara menyimpan sebagai file:**
1.  Salin seluruh teks di dalam blok kode di atas.
2.  Buka editor teks apa pun di komputer Anda (seperti Notepad di Windows, TextEdit di macOS, atau editor kode seperti VS Code).
3.  Tempel teks yang sudah disalin.
4.  Simpan file dengan nama seperti `ringkasan_alur.md` atau `ringkasan_alur.txt`.

## 1. Data Loading

In [ ]:
import pandas as pd

# Load the main training dataset
try:
    df_train = pd.read_csv('application_train.csv')
    print("application_train.csv loaded successfully.")
except FileNotFoundError:
    print("Error: 'application_train.csv' not found. Please upload the file or ensure the path is correct.")
    df_train = None # Set to None to avoid further errors

## 2. Initial Data Exploration

In [ ]:
if df_train is not None:
    print("\n--- First 5 rows of application_train.csv ---")
    display(df_train.head())

    print("\n--- Dataset Information (Data Types and Non-Null Counts) ---")
    df_train.info()

    print("\n--- Missing Values Count ---")
    missing_values = df_train.isnull().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
    display(pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage': (missing_values / len(df_train)) * 100}))

    print("\n--- Descriptive Statistics ---")
    display(df_train.describe())

    print("\n--- Target Variable Distribution ---")
    display(df_train['TARGET'].value_counts(normalize=True) * 100)

## 3. Data Preprocessing

### 3.1. Handling Missing Values

In [ ]:
# Calculate the percentage of missing values for each column
missing_percentage = df_train.isnull().sum() / len(df_train) * 100

# Display columns with a missing value percentage above a threshold (e.g., 50%)
threshold = 50
columns_to_drop = missing_percentage[missing_percentage > threshold].index.tolist()

print(f"Columns to be dropped (missing values > {threshold}%): {len(columns_to_drop)} columns")
print(columns_to_drop)

# Drop these columns from the DataFrame
df_train_interim = df_train.drop(columns=columns_to_drop)
print(f"Number of columns after dropping: {df_train_interim.shape[1]}")

# Update the list of columns that still have missing values
missing_values_after_drop = df_train_interim.isnull().sum()
missing_values_after_drop = missing_values_after_drop[missing_values_after_drop > 0].sort_values(ascending=False)

print("\n--- Missing Values After Column Dropping --- ")
display(pd.DataFrame({'Missing Count': missing_values_after_drop, 'Missing Percentage': (missing_values_after_drop / len(df_train_interim)) * 100}))

# Separate numerical and categorical columns that still have missing values
numerical_cols_with_missing = df_train_interim.select_dtypes(include=['int64', 'float64']).columns
categorical_cols_with_missing = df_train_interim.select_dtypes(include=['object']).columns

# Impute numerical columns with the median
for col in numerical_cols_with_missing:
    if df_train_interim[col].isnull().any():
        median_val = df_train_interim[col].median()
        df_train_interim[col] = df_train_interim[col].fillna(median_val)

# Impute categorical columns with the mode
for col in categorical_cols_with_missing:
    if df_train_interim[col].isnull().any():
        mode_val = df_train_interim[col].mode()[0]
        df_train_interim[col] = df_train_interim[col].fillna(mode_val)

print("\n--- Missing Values Verification After Imputation --- ")
missing_values_final = df_train_interim.isnull().sum()
missing_values_final = missing_values_final[missing_values_final > 0]

if missing_values_final.empty:
    print("No more missing values in the DataFrame.")
else:
    display(pd.DataFrame({'Missing Count': missing_values_final}))

### 3.2. Handling Categorical Variables (One-Hot Encoding)

In [ ]:
# Identify categorical columns
categorical_cols = df_train_interim.select_dtypes(include=['object']).columns

print(f"Categorical columns to be encoded: {len(categorical_cols)} columns")
print(categorical_cols.tolist())

# Perform One-Hot Encoding
df_train_processed = pd.get_dummies(df_train_interim, columns=categorical_cols, dummy_na=False)

print(f"\nNumber of columns after One-Hot Encoding: {df_train_processed.shape[1]}")

# Display the first 5 rows of the DataFrame after encoding
print("\n--- First 5 Rows of DataFrame After One-Hot Encoding ---")
display(df_train_processed.head())

### 3.3. Preparing Data for Modeling (Splitting Features and Target)

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df_train_processed.drop('TARGET', axis=1)
y = df_train_processed['TARGET']

# Handle any remaining non-numeric columns (e.g., booleans)
# Convert booleans to integers (0 or 1)
for col in X.select_dtypes(include=['bool']).columns:
    X[col] = X[col].astype(int)

# Drop the 'SK_ID_CURR' column as it's not used as a feature in modeling
X = X.drop('SK_ID_CURR', axis=1, errors='ignore')

print(f"Number of features (X) after preparation: {X.shape[1]}")
print(f"Size of target variable (y): {y.shape[0]}")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data size (X_train): {X_train.shape}")
print(f"Testing data size (X_test): {X_test.shape}")
print(f"Target distribution in training data:\n{y_train.value_counts(normalize=True)}")
print(f"Target distribution in testing data:\n{y_test.value_counts(normalize=True)}")

## 4. Modeling

### 4.1. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Initialize Logistic Regression model
# Use `class_weight='balanced'` to handle class imbalance
log_reg_model = LogisticRegression(solver='liblinear', random_state=42, class_weight='balanced', max_iter=1000)

# Train the model on the training data
print("Training Logistic Regression model...")
log_reg_model.fit(X_train, y_train)
print("Logistic Regression model training completed.")

# Make probability predictions on the testing data
y_pred_log_reg = log_reg_model.predict_proba(X_test)[:, 1]

# Calculate and display the ROC-AUC score
roc_auc_log_reg = roc_auc_score(y_test, y_pred_log_reg)
print(f"ROC-AUC Score (Logistic Regression): {roc_auc_log_reg:.4f}")

### 4.2. XGBoost Classifier

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Initialize XGBoost Classifier model
# Use `scale_pos_weight` to handle class imbalance
# `scale_pos_weight` is calculated as (number_of_negative_cases / number_of_positive_cases)
positive_cases = y_train.sum()
negative_cases = len(y_train) - positive_cases
scale_pos_weight = negative_cases / positive_cases

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=200, # Initial number of estimators
    learning_rate=0.1,
    max_depth=5,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    scale_pos_weight=scale_pos_weight # Handle class imbalance
)

# Train the model on the training data
print("Training XGBoost model...")
xgb_model.fit(X_train, y_train)
print("XGBoost model training completed.")

# Make probability predictions on the testing data
y_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Calculate and display the ROC-AUC score
roc_auc_xgb = roc_auc_score(y_test, y_pred_xgb)
print(f"ROC-AUC Score (XGBoost): {roc_auc_xgb:.4f}")

### 4.3. Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Initialize Decision Tree Classifier model
# Use `class_weight='balanced'` to handle class imbalance
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# Train the model on the training data
print("Training Decision Tree model...")
dt_model.fit(X_train, y_train)
print("Decision Tree model training completed.")

# Make probability predictions on the testing data
y_pred_dt = dt_model.predict_proba(X_test)[:, 1]

# Calculate and display the ROC-AUC score
roc_auc_dt = roc_auc_score(y_test, y_pred_dt)
print(f"ROC-AUC Score (Decision Tree): {roc_auc_dt:.4f}")

## 5. Model Evaluation Summary

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

print("Starting performance evaluation for each model...")

# --- Logistic Regression ---
# Predictions for Logistic Regression
y_train_pred_log_reg = log_reg_model.predict(X_train)
y_test_pred_log_reg = log_reg_model.predict(X_test)
y_test_pred_proba_log_reg = log_reg_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for Logistic Regression
train_accuracy_log_reg = accuracy_score(y_train, y_train_pred_log_reg) * 100
test_accuracy_log_reg = accuracy_score(y_test, y_test_pred_log_reg) * 100
roc_auc_log_reg_score = roc_auc_score(y_test, y_test_pred_proba_log_reg) * 100 # Convert to percentage

# --- XGBoost Classifier ---
# Predictions for XGBoost
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)
y_test_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for XGBoost
train_accuracy_xgb = accuracy_score(y_train, y_train_pred_xgb) * 100
test_accuracy_xgb = accuracy_score(y_test, y_test_pred_xgb) * 100
roc_auc_xgb_score = roc_auc_score(y_test, y_test_pred_proba_xgb) * 100 # Convert to percentage

# --- Decision Tree Classifier ---
# Predictions for Decision Tree
y_train_pred_dt = dt_model.predict(X_train)
y_test_pred_dt = dt_model.predict(X_test)
y_test_pred_proba_dt = dt_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for Decision Tree
train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt) * 100
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt) * 100
roc_auc_dt_score = roc_auc_score(y_test, y_test_pred_proba_dt) * 100 # Convert to percentage

print("Performance evaluation completed.")

# Create a DataFrame to display the results neatly
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost', 'Decision Tree'],
    'Training Accuracy (%)': [train_accuracy_log_reg, train_accuracy_xgb, train_accuracy_dt],
    'Testing Accuracy (%)': [test_accuracy_log_reg, test_accuracy_xgb, test_accuracy_dt],
    'ROC-AUC Score (%)': [roc_auc_log_reg_score, roc_auc_xgb_score, roc_auc_dt_score]
})

display(results_df)

## 6. Feature Importance from XGBoost Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get feature importances from the XGBoost model
feature_importances = xgb_model.feature_importances_

# Create a DataFrame for visualization
features_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
})

# Sort by importance
features_df = features_df.sort_values(by='Importance', ascending=False)

print("Top 20 Most Important Features according to XGBoost:")
display(features_df.head(20))

# Visualize Top 20 Feature Importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df.head(20), hue='Feature', palette='viridis', legend=False)
plt.title('Top 20 Feature Importance from XGBoost Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 7. Visualizing Key Insights

### 7.1. Insight 1: External Credit Quality and Employment Status

First, let's look at the distribution of `EXT_SOURCE_2` and `EXT_SOURCE_3` (external source scores) compared to the target variable (`TARGET`). Lower scores on `EXT_SOURCE` generally indicate higher risk.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 6))

# Plot EXT_SOURCE_2 vs TARGET
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.violinplot(x='TARGET', y='EXT_SOURCE_2', data=df_train_processed, hue='TARGET', palette='viridis', legend=False)
plt.title('EXT_SOURCE_2 Distribution by Target')
plt.xlabel('Target (0: Repaid, 1: Default)')
plt.ylabel('EXT_SOURCE_2 Score')

# Plot EXT_SOURCE_3 vs TARGET
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.violinplot(x='TARGET', y='EXT_SOURCE_3', data=df_train_processed, hue='TARGET', palette='plasma', legend=False)
plt.title('EXT_SOURCE_3 Distribution by Target')
plt.xlabel('Target (0: Repaid, 1: Default)')
plt.ylabel('EXT_SOURCE_3 Score')

plt.tight_layout()
plt.show()

Next, let's visualize the relationship between `NAME_INCOME_TYPE` (income type) and `FLAG_EMP_PHONE` (ownership of a mobile phone registered as an office phone) with the probability of default (`TARGET`). `FLAG_EMP_PHONE` indicates employment stability.

In [ ]:
plt.figure(figsize=(16, 6))

# Plot NAME_INCOME_TYPE vs TARGET (Mean of Target)
plt.subplot(1, 2, 1)
sns.barplot(x='NAME_INCOME_TYPE', y='TARGET', data=df_train_processed, hue='NAME_INCOME_TYPE', palette='coolwarm', legend=False)
plt.title('Default Rate by Income Type')
plt.xlabel('Income Type')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks(rotation=45, ha='right')

# Plot FLAG_EMP_PHONE vs TARGET (Mean of Target)
plt.subplot(1, 2, 2)
sns.barplot(x='FLAG_EMP_PHONE', y='TARGET', data=df_train_processed, hue='FLAG_EMP_PHONE', palette='rocket', legend=False)
plt.title('Default Rate by Employee Phone Flag (1: Has Phone, 0: No Phone)')
plt.xlabel('FLAG_EMP_PHONE')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks([0, 1], ['No Employee Phone', 'Has Employee Phone'])

plt.tight_layout()
plt.show()

### 7.2. Insight 2: Education and Consumer Demographics

Finally, let's visualize how education level, car ownership, and client region rating (`REGION_RATING_CLIENT_W_CITY`) are related to the probability of default (`TARGET`).

In [ ]:
plt.figure(figsize=(18, 6))

# Plot NAME_EDUCATION_TYPE vs TARGET (Mean of Target)
plt.subplot(1, 3, 1)
sns.barplot(x='NAME_EDUCATION_TYPE', y='TARGET', data=df_train_processed, hue='NAME_EDUCATION_TYPE', palette='crest', legend=False)
plt.title('Default Rate by Education Type')
plt.xlabel('Education Type')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks(rotation=45, ha='right')

# Plot FLAG_OWN_CAR vs TARGET (Mean of Target)
plt.subplot(1, 3, 2)
sns.barplot(x='FLAG_OWN_CAR', y='TARGET', data=df_train_processed, hue='FLAG_OWN_CAR', palette='mako', legend=False)
plt.title('Default Rate by Car Ownership')
plt.xlabel('Owns Car (N: No, Y: Yes)')
plt.ylabel('Default Rate (Mean Target)')

# Plot REGION_RATING_CLIENT_W_CITY vs TARGET (Mean of Target)
plt.subplot(1, 3, 3)
sns.barplot(x='REGION_RATING_CLIENT_W_CITY', y='TARGET', data=df_train_processed, hue='REGION_RATING_CLIENT_W_CITY', palette='rocket', legend=False)
plt.title('Default Rate by Region Rating (Client in City)')
plt.xlabel('Region Rating (Client in City)')
plt.ylabel('Default Rate (Mean Target)')

plt.tight_layout()
plt.show()

## 8. Workflow Summary (Mermaid and Table)

```mermaid
graph TD
    A[Start: Data Input] --> B{Feature Selection};
    B --> B1[Load application_train.csv];
    B1 --> B2[Initial Features: All columns];
    B2 --> B3[Drop SK_ID_CURR];

    B3 --> C{Data Preprocessing};
    C --> C1[Missing Values Handling];
    C1 --> C1_1[Columns with >50% missing values removed];
    C1 --> C1_2[Numerical missing values imputed with Median];
    C1 --> C1_3[Categorical missing values imputed with Mode];

    C1_3 --> C2[Categorical Variables];
    C2 --> C2_1[One-Hot Encoding];

    C2_1 --> C3[Data Splitting];
    C3 --> C3_1[Separate Features X and Target y];
    C3 --> C3_2[Split into Train 80% and Test 20%];
    C3 --> C3_3[Stratified Split for Target Class Balance];

    C3_3 --> D{Modeling Algorithms};
    D --> D1[Logistic Regression];
    D1 --> D1_1[Imbalance Handling: class_weight='balanced'];

    D --> D2[XGBoost Classifier];
    D2 --> D2_1[Imbalance Handling: scale_pos_weight];

    D --> D3[Decision Tree Classifier];
    D3 --> D3_1[Imbalance Handling: class_weight='balanced'];

    D1_1 & D2_1 & D3_1 --> E{Evaluation Metrics};
    E --> E1[ROC-AUC Score];
    E1 --> F[End];
```

**How to save as a file:**
1.  Copy the entire text within the code block above (from `graph TD` to `F[End];`).
2.  Open any text editor on your computer (such as Notepad on Windows, TextEdit on macOS, or a code editor like VS Code).
3.  Paste the copied text.
4.  Save the file with a name like `modeling_flow.mmd` or `modeling_flow.txt`.

### Model Workflow Summary (Table)

| Stage                      | Sub-Stage                         | Detailed Description                                                              |
| :------------------------- | :-------------------------------- | :-------------------------------------------------------------------------------- |
| **1. Data Input & Feature Selection** | **Data Source**                   | `application_train.csv`                                                           |
|                            | **Initial Features**              | All columns from `application_train.csv`                                          |
|                            | **Feature Removal**               | `SK_ID_CURR` column dropped.                                                       |
| **2. Data Preprocessing**  | **Missing Values Handling**       | Columns with `>50%` missing values were dropped. Numerical missing values imputed with Median. Categorical missing values imputed with Mode. |
|                            | **Categorical Variables**         | **One-Hot Encoding** performed on 13 categorical columns.                          |
|                            | **Data Splitting**                | Features (`X`) and Target (`y`) separated. Data split into 80% Train, 20% Test (Stratified Split). |
| **3. Modeling Algorithms** | **Logistic Regression**           | Used `class_weight='balanced'`. ROC-AUC: 0.6425.                          |
|                            | **XGBoost Classifier**            | Used `scale_pos_weight`. ROC-AUC: 0.7567 (best performing).                        |
|                            | **Decision Tree Classifier**      | Used `class_weight='balanced'`. ROC-AUC: 0.5429.                          |
| **4. Evaluation Metrics**  | **ROC-AUC Score**                 | Used for all models as the primary metric.                                 |

**How to save as a file:**
1.  Copy the entire text within the code block above.
2.  Open any text editor on your computer (such as Notepad on Windows, TextEdit on macOS, or a code editor like VS Code).
3.  Paste the copied text.
4.  Save the file with a name like `workflow_summary.md` or `workflow_summary.txt`.

### Ringkasan Alur Pembuatan Model (Tabel)

| Tahap                      | Sub-Tahap                         | Deskripsi Detail                                                                  |
| :------------------------- | :-------------------------------- | :-------------------------------------------------------------------------------- |
| **1. Data Input & Feature Selection** | **Sumber Data**                   | `application_train.csv`                                                           |
|                            | **Fitur Awal**                    | Semua kolom dari `application_train.csv`                                          |
|                            | **Penghapusan Fitur**             | Kolom `SK_ID_CURR` dihapus.                                                       |
| **2. Pra-pemrosesan Data** | **Penanganan Missing Values**     | Kolom dengan `>50%` missing values dihapus. Numerik diimputasi dengan Median. Kategorikal diimputasi dengan Modus. |
|                            | **Variabel Kategorikal**          | Dilakukan **One-Hot Encoding** pada 13 kolom kategorikal.                          |
|                            | **Pembagian Data**                | Fitur (`X`) dan Target (`y`) dipisah. Data dibagi 80% Train, 20% Test (Stratified Split). |
| **3. Algoritma Pemodelan** | **Logistic Regression**           | Menggunakan `class_weight='balanced'`. ROC-AUC: 0.6425.                          |
|                            | **XGBoost Classifier**            | Menggunakan `scale_pos_weight`. ROC-AUC: 0.7567 (terbaik).                        |
|                            | **Decision Tree Classifier**      | Menggunakan `class_weight='balanced'`. ROC-AUC: 0.5429.                          |
| **4. Metrik Evaluasi**     | **ROC-AUC Score**                 | Digunakan untuk semua model sebagai metrik utama.                                 |

### Ringkasan Alur Pembuatan Model (Tabel)

| Tahap                      | Sub-Tahap                         | Deskripsi Detail                                                                  |
| :------------------------- | :-------------------------------- | :-------------------------------------------------------------------------------- |
| **1. Data Input & Feature Selection** | **Sumber Data**                   | `application_train.csv`                                                           |
|                            | **Fitur Awal**                    | Semua kolom dari `application_train.csv`                                          |
|                            | **Penghapusan Fitur**             | Kolom `SK_ID_CURR` dihapus.                                                       |
| **2. Pra-pemrosesan Data** | **Penanganan Missing Values**     | Kolom dengan `>50%` missing values dihapus. Numerik diimputasi dengan Median. Kategorikal diimputasi dengan Modus. |
|                            | **Variabel Kategorikal**          | Dilakukan **One-Hot Encoding** pada 13 kolom kategorikal.                          |
|                            | **Pembagian Data**                | Fitur (`X`) dan Target (`y`) dipisah. Data dibagi 80% Train, 20% Test (Stratified Split). |
| **3. Algoritma Pemodelan** | **Logistic Regression**           | Menggunakan `class_weight='balanced'`. ROC-AUC: 0.6425.                          |
|                            | **XGBoost Classifier**            | Menggunakan `scale_pos_weight`. ROC-AUC: 0.7567 (terbaik).                        |
|                            | **Decision Tree Classifier**      | Menggunakan `class_weight='balanced'`. ROC-AUC: 0.5429.                          |
| **4. Metrik Evaluasi**     | **ROC-AUC Score**                 | Digunakan untuk semua model sebagai metrik utama.                                 |

**Cara menyimpan sebagai file:**
1.  Salin seluruh teks di dalam blok kode di atas.
2.  Buka editor teks apa pun di komputer Anda (seperti Notepad di Windows, TextEdit di macOS, atau editor kode seperti VS Code).
3.  Tempel teks yang sudah disalin.
4.  Simpan file dengan nama seperti `ringkasan_alur.md` atau `ringkasan_alur.txt`.

Now, let's perform some initial data exploration to understand the dataset's structure and content.

### Langkah 1: Menangani Nilai yang Hilang (Missing Values)

In [ ]:
# Menghitung persentase nilai hilang untuk setiap kolom
missing_percentage = df_train.isnull().sum() / len(df_train) * 100

# Menampilkan kolom dengan persentase nilai hilang lebih dari ambang batas (misal 50%)
threshold = 50
columns_to_drop = missing_percentage[missing_percentage > threshold].index.tolist()

print(f"Kolom yang akan dihapus (nilai hilang > {threshold}%): {len(columns_to_drop)} kolom")
print(columns_to_drop)

# Menghapus kolom-kolom tersebut dari DataFrame
df_train_cleaned = df_train.drop(columns=columns_to_drop)
print(f"Jumlah kolom setelah penghapusan: {df_train_cleaned.shape[1]}")

# Memperbarui daftar kolom yang memiliki nilai hilang
missing_values_after_drop = df_train_cleaned.isnull().sum()
missing_values_after_drop = missing_values_after_drop[missing_values_after_drop > 0].sort_values(ascending=False)

print("\n--- Nilai Hilang Setelah Penghapusan Kolom --- ")
display(pd.DataFrame({'Missing Count': missing_values_after_drop, 'Missing Percentage': (missing_values_after_drop / len(df_train_cleaned)) * 100}))


Setelah menghapus kolom dengan banyak nilai hilang, kita perlu menangani nilai hilang yang tersisa di kolom lain. Untuk kolom numerik, kita bisa mengimputasi dengan median atau rata-rata. Untuk kolom kategorikal, kita bisa mengimputasi dengan modus (nilai paling sering muncul) atau menandainya sebagai 'Missing'.

### Langkah 2: Mengelola Variabel Kategorikal (One-Hot Encoding)

In [ ]:
# Identifikasi kolom kategorikal
categorical_cols = df_train.select_dtypes(include=['object']).columns

print(f"Kolom kategorikal yang akan di-encode: {len(categorical_cols)} kolom")
print(categorical_cols.tolist())

# Lakukan One-Hot Encoding
df_train_encoded = pd.get_dummies(df_train, columns=categorical_cols, dummy_na=False)

print(f"\nJumlah kolom setelah One-Hot Encoding: {df_train_encoded.shape[1]}")

# Tampilkan 5 baris pertama DataFrame setelah encoding
print("\n--- 5 Baris Pertama DataFrame Setelah One-Hot Encoding ---")
display(df_train_encoded.head())

df_train = df_train_encoded.copy()

In [ ]:
# Memisahkan kolom numerik dan kategorikal yang masih memiliki nilai hilang
numerical_cols_with_missing = df_train_cleaned.select_dtypes(include=['int64', 'float64']).columns
categorical_cols_with_missing = df_train_cleaned.select_dtypes(include=['object']).columns

# Imputasi kolom numerik dengan median
for col in numerical_cols_with_missing:
    if df_train_cleaned[col].isnull().any():
        median_val = df_train_cleaned[col].median()
        df_train_cleaned[col] = df_train_cleaned[col].fillna(median_val)
        # print(f"Imputasi kolom numerik '{col}' dengan median: {median_val}")

# Imputasi kolom kategorikal dengan modus (mode)
for col in categorical_cols_with_missing:
    if df_train_cleaned[col].isnull().any():
        mode_val = df_train_cleaned[col].mode()[0]
        df_train_cleaned[col] = df_train_cleaned[col].fillna(mode_val)
        # print(f"Imputasi kolom kategorikal '{col}' dengan modus: {mode_val}")

print("\n--- Verifikasi Nilai Hilang Setelah Imputasi --- ")
missing_values_final = df_train_cleaned.isnull().sum()
missing_values_final = missing_values_final[missing_values_final > 0]

if missing_values_final.empty:
    print("Tidak ada lagi nilai hilang di DataFrame.")
else:
    display(pd.DataFrame({'Missing Count': missing_values_final}))

df_train = df_train_cleaned.copy()


In [ ]:
if df_train is not None:
    print("\n--- First 5 rows of application_train.csv ---")
    display(df_train.head())

    print("\n--- Dataset Information (Data Types and Non-Null Counts) ---")
    df_train.info()

    print("\n--- Missing Values Count ---")
    missing_values = df_train.isnull().sum()
    missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
    display(pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage': (missing_values / len(df_train)) * 100}))

    print("\n--- Descriptive Statistics ---")
    display(df_train.describe())

    print("\n--- Target Variable Distribution ---")
    display(df_train['TARGET'].value_counts(normalize=True) * 100)


### Langkah 4: Pemodelan - Logistic Regression

### Langkah 6: Pemodelan - Random Forest Classifier dengan Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

# Inisialisasi model Random Forest
# Menggunakan `class_weight='balanced'` untuk menangani ketidakseimbangan kelas
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')

# Definisikan parameter grid untuk hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300],  # Jumlah pohon dalam forest
    'max_depth': [10, 20, None],      # Kedalaman maksimum pohon
    'min_samples_split': [2, 5],      # Jumlah minimum sampel yang dibutuhkan untuk membagi node internal
    'min_samples_leaf': [1, 2]        # Jumlah minimum sampel yang dibutuhkan untuk menjadi node daun
}

# Inisialisasi GridSearchCV
# Scoring menggunakan 'roc_auc' karena ini adalah metrik evaluasi yang diminta
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid,
                           scoring='roc_auc', cv=3, verbose=2, n_jobs=-1)

print("Melakukan hyperparameter tuning untuk Random Forest...")
grid_search.fit(X_train, y_train)
print("Hyperparameter tuning selesai.")

# Dapatkan model terbaik dari Grid Search
best_rf_model = grid_search.best_estimator_

print(f"Parameter terbaik untuk Random Forest: {grid_search.best_params_}")

# Melakukan prediksi probabilitas pada data pengujian dengan model terbaik
y_pred_rf = best_rf_model.predict_proba(X_test)[:, 1]

# Menghitung dan menampilkan ROC-AUC score
roc_auc_rf = roc_auc_score(y_test, y_pred_rf)
print(f"ROC-AUC Score (Random Forest dengan Tuning): {roc_auc_rf:.4f}")

### Langkah 7: Pemodelan - Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Inisialisasi model Decision Tree Classifier
# Menggunakan `class_weight='balanced'` untuk menangani ketidakseimbangan kelas
dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# Melatih model pada data pelatihan
print("Melatih model Decision Tree...")
dt_model.fit(X_train, y_train)
print("Model Decision Tree selesai dilatih.")

# Melakukan prediksi probabilitas pada data pengujian
y_pred_dt = dt_model.predict_proba(X_test)[:, 1]

# Menghitung dan menampilkan ROC-AUC score
roc_auc_dt = roc_auc_score(y_test, y_pred_dt)
print(f"ROC-AUC Score (Decision Tree): {roc_auc_dt:.4f}")

### Langkah 5: Pemodelan - XGBoost Classifier

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Inisialisasi model XGBoost Classifier
# Menggunakan `scale_pos_weight` untuk menangani ketidakseimbangan kelas
# Nilai `scale_pos_weight` dihitung sebagai (jumlah_negatif / jumlah_positif)
positive_cases = y_train.sum()
negative_cases = len(y_train) - positive_cases
scale_pos_weight = negative_cases / positive_cases

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=200, # Jumlah estimator awal
    learning_rate=0.1,
    max_depth=5,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    scale_pos_weight=scale_pos_weight # Mengatasi ketidakseimbangan kelas
)

# Melatih model pada data pelatihan
print("Melatih model XGBoost...")
xgb_model.fit(X_train, y_train)
print("Model XGBoost selesai dilatih.")

# Melakukan prediksi probabilitas pada data pengujian
y_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Menghitung dan menampilkan ROC-AUC score
roc_auc_xgb = roc_auc_score(y_test, y_pred_xgb)
print(f"ROC-AUC Score (XGBoost): {roc_auc_xgb:.4f}")

### Ringkasan Performa Model

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

print("Memulai evaluasi performa untuk setiap model...")

# --- Logistic Regression ---
# Predictions for Logistic Regression
y_train_pred_log_reg = log_reg_model.predict(X_train)
y_test_pred_log_reg = log_reg_model.predict(X_test)
y_test_pred_proba_log_reg = log_reg_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for Logistic Regression
train_accuracy_log_reg = accuracy_score(y_train, y_train_pred_log_reg) * 100
test_accuracy_log_reg = accuracy_score(y_test, y_test_pred_log_reg) * 100
roc_auc_log_reg_score = roc_auc_score(y_test, y_test_pred_proba_log_reg) * 100 # Convert to percentage

# --- XGBoost Classifier ---
# Predictions for XGBoost
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)
y_test_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for XGBoost
train_accuracy_xgb = accuracy_score(y_train, y_train_pred_xgb) * 100
test_accuracy_xgb = accuracy_score(y_test, y_test_pred_xgb) * 100
roc_auc_xgb_score = roc_auc_score(y_test, y_test_pred_proba_xgb) * 100 # Convert to percentage

# --- Decision Tree Classifier ---
# Predictions for Decision Tree
y_train_pred_dt = dt_model.predict(X_train)
y_test_pred_dt = dt_model.predict(X_test)
y_test_pred_proba_dt = dt_model.predict_proba(X_test)[:, 1] # ROC-AUC uses probabilities

# Calculate metrics for Decision Tree
train_accuracy_dt = accuracy_score(y_train, y_train_pred_dt) * 100
test_accuracy_dt = accuracy_score(y_test, y_test_pred_dt) * 100
roc_auc_dt_score = roc_auc_score(y_test, y_test_pred_proba_dt) * 100 # Convert to percentage

print("Evaluasi performa selesai.")

# Create a DataFrame to display the results neatly
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost', 'Decision Tree'],
    'Training Accuracy (%)': [train_accuracy_log_reg, train_accuracy_xgb, train_accuracy_dt],
    'Testing Accuracy (%)': [test_accuracy_log_reg, test_accuracy_xgb, test_accuracy_dt],
    'ROC-AUC Score (%)': [roc_auc_log_reg_score, roc_auc_xgb_score, roc_auc_dt_score]
})

display(results_df)

### Analisis Feature Importance dari Model XGBoost

### Visualisasi Insight 1: Kualitas Kredit Eksternal dan Status Pekerjaan

Pertama, mari kita lihat distribusi `EXT_SOURCE_2` dan `EXT_SOURCE_3` (skor sumber eksternal) dibandingkan dengan variabel target (`TARGET`). Skor yang lebih rendah pada `EXT_SOURCE` umumnya mengindikasikan risiko yang lebih tinggi.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 6))

# Plot EXT_SOURCE_2 vs TARGET
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.violinplot(x='TARGET', y='EXT_SOURCE_2', data=df_train_cleaned, hue='TARGET', palette='viridis', legend=False)
plt.title('EXT_SOURCE_2 Distribution by Target')
plt.xlabel('Target (0: Repaid, 1: Default)')
plt.ylabel('EXT_SOURCE_2 Score')

# Plot EXT_SOURCE_3 vs TARGET
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.violinplot(x='TARGET', y='EXT_SOURCE_3', data=df_train_cleaned, hue='TARGET', palette='plasma', legend=False)
plt.title('EXT_SOURCE_3 Distribution by Target')
plt.xlabel('Target (0: Repaid, 1: Default)')
plt.ylabel('EXT_SOURCE_3 Score')

plt.tight_layout()
plt.show()

Selanjutnya, mari kita visualisasikan hubungan antara `NAME_INCOME_TYPE` (tipe pendapatan) dan `FLAG_EMP_PHONE` (kepemilikan telepon seluler yang terdaftar sebagai telepon kantor) dengan probabilitas gagal bayar (`TARGET`). `FLAG_EMP_PHONE` mengindikasikan stabilitas pekerjaan.

In [ ]:
plt.figure(figsize=(16, 6))

# Plot NAME_INCOME_TYPE vs TARGET (Mean of Target)
plt.subplot(1, 2, 1)
sns.barplot(x='NAME_INCOME_TYPE', y='TARGET', data=df_train_cleaned, hue='NAME_INCOME_TYPE', palette='coolwarm', legend=False)
plt.title('Default Rate by Income Type')
plt.xlabel('Income Type')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks(rotation=45, ha='right')

# Plot FLAG_EMP_PHONE vs TARGET (Mean of Target)
plt.subplot(1, 2, 2)
sns.barplot(x='FLAG_EMP_PHONE', y='TARGET', data=df_train_cleaned, hue='FLAG_EMP_PHONE', palette='rocket', legend=False)
plt.title('Default Rate by Employee Phone Flag (1: Has Phone, 0: No Phone)')
plt.xlabel('FLAG_EMP_PHONE')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks([0, 1], ['No Employee Phone', 'Has Employee Phone'])

plt.tight_layout()
plt.show()

### Visualisasi Insight 2: Pendidikan dan Demografi Konsumen

Terakhir, mari kita visualisasikan bagaimana tingkat pendidikan, kepemilikan mobil, dan rating wilayah klien (`REGION_RATING_CLIENT_W_CITY`) berhubungan dengan probabilitas gagal bayar (`TARGET`).

In [ ]:
plt.figure(figsize=(18, 6))

# Plot NAME_EDUCATION_TYPE vs TARGET (Mean of Target)
plt.subplot(1, 3, 1)
sns.barplot(x='NAME_EDUCATION_TYPE', y='TARGET', data=df_train_cleaned, hue='NAME_EDUCATION_TYPE', palette='crest', legend=False)
plt.title('Default Rate by Education Type')
plt.xlabel('Education Type')
plt.ylabel('Default Rate (Mean Target)')
plt.xticks(rotation=45, ha='right')

# Plot FLAG_OWN_CAR vs TARGET (Mean of Target)
plt.subplot(1, 3, 2)
sns.barplot(x='FLAG_OWN_CAR', y='TARGET', data=df_train_cleaned, hue='FLAG_OWN_CAR', palette='mako', legend=False)
plt.title('Default Rate by Car Ownership')
plt.xlabel('Owns Car (N: No, Y: Yes)')
plt.ylabel('Default Rate (Mean Target)')

# Plot REGION_RATING_CLIENT_W_CITY vs TARGET (Mean of Target)
plt.subplot(1, 3, 3)
sns.barplot(x='REGION_RATING_CLIENT_W_CITY', y='TARGET', data=df_train_cleaned, hue='REGION_RATING_CLIENT_W_CITY', palette='rocket', legend=False)
plt.title('Default Rate by Region Rating (Client in City)')
plt.xlabel('Region Rating (Client in City)')
plt.ylabel('Default Rate (Mean Target)')

plt.tight_layout()
plt.show()

### Ringkasan Alur Pembuatan Model

Berikut adalah gambaran visual dari langkah-langkah utama yang telah kita lakukan dalam membangun model prediksi risiko kredit:

---

#### 1. Data Input & Feature Selection

*   **Sumber Data:** `application_train.csv`
*   **Fitur Awal:** Semua kolom dari `application_train.csv`
*   **Penghapusan:** Kolom `SK_ID_CURR` dihapus.

---

#### 2. Pra-pemrosesan Data

*   **Penanganan Missing Values:**
    *   Kolom dengan `>50%` missing values **dihapus**.
    *   Missing values numerik diimputasi dengan **Median**.
    *   Missing values kategorikal diimputasi dengan **Modus**.
*   **Variabel Kategorikal:** Dilakukan **One-Hot Encoding**.
*   **Pembagian Data:**
    *   Fitur (`X`) dan Target (`y`) dipisah.
    *   Data dibagi menjadi **Train (80%)** dan **Test (20%)**.
    *   Menggunakan strategi **Stratified Split** untuk menjaga proporsi kelas target.

---

#### 3. Algoritma Pemodelan

*   **Logistic Regression**
    *   Penanganan Imbalance: `class_weight='balanced'`
*   **XGBoost Classifier**
    *   Penanganan Imbalance: `scale_pos_weight`
*   **Decision Tree Classifier**
    *   Penanganan Imbalance: `class_weight='balanced'`

---

#### 4. Metrik Evaluasi

*   **ROC-AUC Score** (digunakan untuk semua model)

---

Visualisasi ini memberikan representasi singkat namun padat tentang bagaimana data diolah dan model dibangun, dimulai dari input hingga metrik evaluasi akhir.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Dapatkan feature importance dari model XGBoost
feature_importances = xgb_model.feature_importances_

# Buat DataFrame untuk visualisasi
features_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
})

# Urutkan berdasarkan importance
features_df = features_df.sort_values(by='Importance', ascending=False)

print("Top 20 Fitur Paling Penting menurut XGBoost:")
display(features_df.head(20))

# Visualisasi Top 20 Feature Importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df.head(20), hue='Feature', palette='viridis', legend=False)
plt.title('Top 20 Feature Importance dari Model XGBoost')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Inisialisasi model Logistic Regression
# Karena dataset tidak seimbang, kita akan menggunakan 'balanced' untuk class_weight
# Atau bisa juga dengan mengatur parameter solver dan max_iter
log_reg_model = LogisticRegression(solver='liblinear', random_state=42, class_weight='balanced', max_iter=1000)

# Melatih model pada data pelatihan
print("Melatih model Logistic Regression...")
log_reg_model.fit(X_train, y_train)
print("Model Logistic Regression selesai dilatih.")

# Melakukan prediksi probabilitas pada data pengujian
y_pred_log_reg = log_reg_model.predict_proba(X_test)[:, 1]

# Menghitung dan menampilkan ROC-AUC score
roc_auc_log_reg = roc_auc_score(y_test, y_pred_log_reg)
print(f"ROC-AUC Score (Logistic Regression): {roc_auc_log_reg:.4f}")

### Langkah 3: Mempersiapkan Data untuk Pemodelan

In [ ]:
from sklearn.model_selection import train_test_split

# Memisahkan fitur (X) dan target (y)
X = df_train.drop('TARGET', axis=1)
y = df_train['TARGET']

# Menangani kolom non-numerik yang mungkin tersisa setelah one-hot encoding (misal, jika ada nilai True/False)
# Konversi boolean menjadi integer (0 atau 1)
for col in X.select_dtypes(include=['bool']).columns:
    X[col] = X[col].astype(int)

# Memastikan semua kolom adalah numerik setelah konversi
# Jika masih ada kolom non-numerik, kita perlu menanganinya. Untuk sementara kita anggap sudah beres.
# Jika ada ID kolom yang tidak perlu untuk modelling (misal SK_ID_CURR), bisa dihapus di sini
X = X.drop('SK_ID_CURR', axis=1, errors='ignore')

print(f"Jumlah fitur (X) setelah persiapan: {X.shape[1]}")
print(f"Ukuran variabel target (y): {y.shape[0]}")

# Membagi data menjadi set pelatihan dan pengujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Ukuran data pelatihan (X_train): {X_train.shape}")
print(f"Ukuran data pengujian (X_test): {X_test.shape}")
print(f"Distribusi target di data pelatihan:\n{y_train.value_counts(normalize=True)}")
print(f"Distribusi target di data pengujian:\n{y_test.value_counts(normalize=True)}")

#### Langkah-langkah Eksplorasi Data Awal:

1.  **Memuat Dataset Utama:** Pertama, kita memuat file `application_train.csv` ke dalam sebuah DataFrame pandas. Ini adalah langkah awal yang krusial untuk dapat bekerja dengan data.
2.  **Menampilkan 5 Baris Pertama:** Kita melihat 5 baris pertama dari DataFrame untuk mendapatkan gambaran awal tentang struktur data dan beberapa nilai contoh.
3.  **Informasi Dataset (Tipe Data & Jumlah Non-Null):** Menggunakan `df_train.info()`, kita memeriksa tipe data setiap kolom dan jumlah nilai non-null. Ini membantu kita mengidentifikasi kolom yang mungkin memiliki nilai yang hilang atau tipe data yang tidak sesuai.
4.  **Menghitung Nilai yang Hilang:** Kita menghitung jumlah dan persentase nilai yang hilang di setiap kolom. Informasi ini sangat penting untuk tahap pembersihan data selanjutnya.
5.  **Statistik Deskriptif:** Kita menampilkan statistik deskriptif dasar (seperti rata-rata, standar deviasi, min, maks, kuartil) untuk kolom-kolom numerik. Ini memberikan ringkasan cepat tentang distribusi data.
6.  **Distribusi Variabel Target:** Kita memeriksa distribusi variabel `TARGET` untuk melihat proporsi kelas (pinjaman dilunasi vs. tidak dilunasi). Ini penting karena dataset yang tidak seimbang mungkin memerlukan penanganan khusus dalam pemodelan.